In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn PyGithub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.7 MB/s eta 0:00:00


In [2]:
!wget -q https://github.com/adoptium/temurin22-binaries/releases/download/jdk-22.0.2%2B9/OpenJDK22U-jdk_x64_linux_hotspot_22.0.2_9.tar.gz
!tar -xzf OpenJDK22U-jdk_x64_linux_hotspot_22.0.2_9.tar.gz
!mv jdk-22.0.2+9 /usr/local/jdk-22
!sudo update-alternatives --install /usr/bin/java java /usr/local/jdk-22/bin/java 1 -q
!sudo update-alternatives --set java /usr/local/jdk-22/bin/java
import os
os.environ['JAVA_HOME'] = '/usr/local/jdk-22'
os.environ['PATH'] = '/usr/local/jdk-22/bin:' + os.environ['PATH']
!java -version

update-alternatives: error: unknown argument '-q'
update-alternatives: error: alternative /usr/local/jdk-22/bin/java for java not registered; not setting
openjdk version "22.0.2" 2024-07-16
OpenJDK Runtime Environment Temurin-22.0.2+9 (build 22.0.2+9)
OpenJDK 64-Bit Server VM Temurin-22.0.2+9 (build 22.0.2+9, mixed mode, sharing)


In [3]:
!git clone https://github.com/mauricioaniche/ck.git /content/ck -q
%cd /content/ck
!./mvnw clean package -Dmaven.javadoc.skip=true -DskipTests -q
import glob
ck_jars = glob.glob('/content/ck/target/ck-*-jar-with-dependencies.jar')
CK_JAR = ck_jars[0]
print(f'CK jar: {CK_JAR}')

/content/ck
--2026-06-05 17:18:00--  https://repo.maven.apache.org/maven2/io/takari/maven-wrapper/0.5.5/maven-wrapper-0.5.5.jar
Resolving repo.maven.apache.org (repo.maven.apache.org)... 104.18.18.12, 104.18.19.12, 2606:4700::6812:130c, ...
Connecting to repo.maven.apache.org (repo.maven.apache.org)|104.18.18.12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 50710 (50K) [application/java-archive]
Saving to: ‘/content/ck/.mvn/wrapper/maven-wrapper.jar’

/content/ck/.mvn/wr 100%[===================>]  49.52K  --.-KB/s    in 0.002s  

2026-06-05 17:18:00 (30.3 MB/s) - ‘/content/ck/.mvn/wrapper/maven-wrapper.jar’ saved [50710/50710]

CK jar: /content/ck/target/ck-0.7.1-SNAPSHOT-jar-with-dependencies.jar


In [4]:
REPOS = {
    'BGPP'                 : 'https://github.com/itu-bswu/BGPP.git',
    'Rivfader'             : 'https://github.com/Tetha/Rivfader.git',
    'Sugarscape'           : 'https://github.com/wsmirnow/Sugarscape.git',
    'XMPP-Client'          : 'https://github.com/lboynton/XMPP-Client.git',
    'cmu-commons'          : 'https://github.com/sagemintblue/cmu-commons.git',
    'interviews'           : 'https://github.com/kdn251/interviews.git',
    'java-design-patterns' : 'https://github.com/iluwatar/java-design-patterns.git',
    'mall'                 : 'https://github.com/macrozheng/mall.git',
    'RgB'                  : 'https://github.com/fedo/RgB.git',
}

REPOS_ROOT = '/content/ck/repos_java'
import os, subprocess
os.makedirs(REPOS_ROOT, exist_ok=True)

for name, url in REPOS.items():
    dest = os.path.join(REPOS_ROOT, name)
    if os.path.exists(dest):
        print(f'Já existe: {name}')
    else:
        print(f'Clonando: {name}')
        subprocess.run(['git', 'clone', '--depth=1', url, dest],
                       capture_output=True)
        print(f'  {name}')

print('\nClonagem concluída')

Clonando: BGPP
  BGPP
Clonando: Rivfader
  Rivfader
Clonando: Sugarscape
  Sugarscape
Clonando: XMPP-Client
  XMPP-Client
Clonando: cmu-commons
  cmu-commons
Clonando: interviews
  interviews
Clonando: java-design-patterns
  java-design-patterns
Clonando: mall
  mall
Clonando: RgB
  RgB

Clonagem concluída


In [5]:
import subprocess, os, glob

CK_JAR     = glob.glob('/content/ck/target/ck-*-jar-with-dependencies.jar')[0]
REPOS_ROOT = '/content/ck/repos_java'
CK_OUTPUT  = '/content/ck_output'
os.makedirs(CK_OUTPUT, exist_ok=True)

for repo_name in os.listdir(REPOS_ROOT):
    repo_path   = os.path.join(REPOS_ROOT, repo_name)
    output_dir  = os.path.join(CK_OUTPUT, repo_name)
    os.makedirs(output_dir, exist_ok=True)

    # Verifica se já foi processado
    if os.path.exists(os.path.join(output_dir, 'class.csv')):
        print(f'⏭️  Já processado: {repo_name}')
        continue

    print(f'Analisando: {repo_name}')
    result = subprocess.run(
        ['java', '-jar', CK_JAR,
         repo_path,   # pasta raiz do projeto
         'true',      # inclui inner classes
         '0',         # threshold 0
         'false',
         output_dir],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        csv = os.path.join(output_dir, 'class.csv')
        if os.path.exists(csv):
            import pandas as pd
            n = pd.read_csv(csv).shape[0]
            print(f'   {n} classes extraídas')
        else:
            print(f'   class.csv não gerado')
    else:
        print(f'   Erro: {result.stderr[:200]}')

print('\nCK concluído')

Analisando: BGPP
   39 classes extraídas
Analisando: Sugarscape
   12 classes extraídas
Analisando: interviews
   257 classes extraídas
Analisando: cmu-commons
   315 classes extraídas
Analisando: Rivfader
   143 classes extraídas
Analisando: RgB
   89 classes extraídas
Analisando: mall
   757 classes extraídas
Analisando: XMPP-Client
   1223 classes extraídas
Analisando: java-design-patterns
   2047 classes extraídas

CK concluído


In [6]:
import pandas as pd, glob, os

CK_OUTPUT  = '/content/ck_output'
REPOS_ROOT = '/content/ck/repos_java'

frames = []

for csv_path in sorted(glob.glob(f'{CK_OUTPUT}/**/class.csv', recursive=True)):
    # Nome do projeto = nome da pasta dentro de ck_output
    project_name = csv_path.split('/')[-2]
    try:
        df_part = pd.read_csv(csv_path)
        df_part.columns = df_part.columns.str.strip()
        df_part['project'] = project_name
        frames.append(df_part)
        print(f'{project_name:<30} {df_part.shape[0]:>6} classes')
    except Exception as e:
        print(f'Erro em {csv_path}: {e}')

df_metrics = pd.concat(frames, ignore_index=True)

print(f'\nShape total: {df_metrics.shape}')
print(f'Projetos  : {df_metrics["project"].nunique()}')
print(f'Colunas   : {list(df_metrics.columns)}')

df_metrics.to_csv('/content/dataset_metrics.csv', index=False)
print('\ndataset_metrics.csv salvo')

BGPP                               39 classes
RgB                                89 classes
Rivfader                          143 classes
Sugarscape                         12 classes
XMPP-Client                      1223 classes
cmu-commons                       315 classes
interviews                        257 classes
java-design-patterns             2047 classes
mall                              757 classes

Shape total: (4882, 53)
Projetos  : 9
Colunas   : ['file', 'class', 'type', 'cbo', 'cboModified', 'fanin', 'fanout', 'wmc', 'dit', 'noc', 'rfc', 'lcom', 'lcom*', 'tcc', 'lcc', 'totalMethodsQty', 'staticMethodsQty', 'publicMethodsQty', 'privateMethodsQty', 'protectedMethodsQty', 'defaultMethodsQty', 'visibleMethodsQty', 'abstractMethodsQty', 'finalMethodsQty', 'synchronizedMethodsQty', 'totalFieldsQty', 'staticFieldsQty', 'publicFieldsQty', 'privateFieldsQty', 'protectedFieldsQty', 'defaultFieldsQty', 'finalFieldsQty', 'synchronizedFieldsQty', 'nosi', 'loc', 'returnQty', 'loopQty